In [2]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from IPython.display import FileLink

models_path = '/kaggle/input/datasets/vvvvadim/models/models'

models_list = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for fold in range(5):
    model = models.efficientnet_b1(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, 5)
    model.load_state_dict(torch.load(os.path.join(models_path, f'efficientnet_b1{fold}_best.pth'), map_location=device))
    model = model.to(device)
    model.eval()
    models_list.append(model)
    print('Model has been loaded')
    

transform_val = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
)


class CassavaTestDataset(Dataset):
    def __init__(self, data, data_root, transforms=None):
        self.data = data
        self.data_root = data_root
        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image_id = self.data[index]
        img_path = os.path.join(self.data_root, image_id)
        image = Image.open(img_path).convert('RGB')
        if self.transforms:
            image = self.transforms(image)
        return image, image_id


path_test = '/kaggle/input/competitions/cassava-leaf-disease-classification/test_images/'

test_image_files = os.listdir(path_test)
print(len(test_image_files))
print(test_image_files[:5])

test_df = pd.DataFrame({'image_id': test_image_files})
print(test_df.head())

test_ds = CassavaTestDataset(test_image_files, path_test, transforms=transform_val)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)



all_preds = []
all_images_ids = []

with torch.no_grad():
    for images, im_ids in test_loader:
        images = images.to(device)

        batch_probs = []
        
        for model in models_list:
            outputs = model(images)
            preds = torch.softmax(outputs, dim=1)
            batch_probs.append(preds)
        
        avg_probs = torch.mean(torch.stack(batch_probs), dim=0)
        _, preds = avg_probs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_images_ids.extend(im_ids)

submission = pd.DataFrame({'image_id': all_images_ids, 'label': all_preds})
submission = submission.sort_values('image_id').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)
print(submission.head())
display(FileLink('submission.csv'))

Model has been loaded
Model has been loaded
Model has been loaded
Model has been loaded
Model has been loaded
1
['2216849948.jpg']
         image_id
0  2216849948.jpg
         image_id  label
0  2216849948.jpg      2


/kaggle/working/submission.csv